In [58]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import math

device  =  'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {device}')

#CONSTANTS:
batch_size = 64
block_size = 256
embed_count = 128
dropout_percent = 0.2

Device: cuda


In [59]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-25 16:39:06--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.13’

input.txt.13        100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-10-25 16:39:06 (31.2 MB/s) - ‘input.txt.13’ saved [1115394/1115394]



In [60]:
import triton
import triton.language as tl
import math
import torch

@triton.jit
def flash_attention_kernel(
    Q_ptr, # M D
    K_ptr, # N D
    V_ptr, # N D
    O_ptr, # M D
    LSE_ptr,
    M, N,
    stride_qm, stride_qd, stride_qb, stride_qh,
    stride_kn, stride_kd, stride_kb, stride_kh,
    stride_vn, stride_vd, stride_vb, stride_vh,
    stride_om, stride_od, stride_ob, stride_oh,
    stride_lseb, stride_lseh, stride_lsem,
    div_factor,
    HEADS_PER_BATCH : tl.constexpr,
    BLOCK_M : tl.constexpr,
    BLOCK_N : tl.constexpr,
    WHOLE_D : tl.constexpr
):
  pid = tl.program_id(0)
  pid_bh = tl.program_id(1)

  current_batch = pid_bh // HEADS_PER_BATCH
  current_head = pid_bh % HEADS_PER_BATCH

  Q_ptr = Q_ptr + current_batch * stride_qb + current_head * stride_qh
  K_ptr = K_ptr + current_batch * stride_kb + current_head * stride_kh
  V_ptr = V_ptr + current_batch * stride_vb + current_head * stride_vh
  O_ptr = O_ptr + current_batch * stride_ob + current_head * stride_oh
  LSE_ptr = LSE_ptr + current_batch * stride_lseb + current_head * stride_lseh

  offsets_m = BLOCK_M * pid + tl.arange(0 , BLOCK_M)
  offsets_n = tl.arange(0 , BLOCK_N)
  offsets_d = tl.arange(0, WHOLE_D)

  mask_q = offsets_m < M

  offsets_matrix_Q = offsets_m[:,None] * stride_qm + offsets_d[None,:] * stride_qd
  q_block = tl.load(Q_ptr + offsets_matrix_Q, mask_q[:, None], other = 0.0)

  offsets_matrix_K = offsets_n[None,:] * stride_kn + offsets_d[:,None] * stride_kd
  offsets_matrix_V = offsets_n[:,None] * stride_vn + offsets_d[None,:] * stride_vd

  prev_max_per_row = tl.zeros([BLOCK_M], dtype=tl.float32) - float('inf')
  denominator_per_row = tl.zeros([BLOCK_M], dtype=tl.float32)

  accumulator = tl.zeros([BLOCK_M, WHOLE_D], dtype=tl.float32)

  end_n = tl.minimum((pid+1)*BLOCK_M, N)

  for n in range(0, end_n, BLOCK_N):
    mask_kv = offsets_n < (N-n)
    causal_mask = (offsets_m[:, None]>=(n + offsets_n)[None, :]) & mask_kv[None, :]

    k_block = tl.load(K_ptr + offsets_matrix_K, mask_kv[None, :], other = 0.0 )
    v_block = tl.load(V_ptr + offsets_matrix_V, mask_kv[:, None], other = 0.0)

    qk_block = tl.dot(q_block,k_block)*div_factor
    qk_block = tl.where(causal_mask, qk_block, -float('inf'))

    temp_max_per_row = tl.max(qk_block,axis=1)

    current_max_per_row = tl.maximum(temp_max_per_row, prev_max_per_row)

    score = tl.exp(qk_block-current_max_per_row[:, None])

    alpha_per_row = tl.exp(prev_max_per_row-current_max_per_row)

    denominator_per_row = denominator_per_row*alpha_per_row + tl.sum(score, axis = -1)

    p = tl.dot(score.to(v_block.dtype), v_block)

    accumulator = accumulator*alpha_per_row[:, None] + p

    prev_max_per_row = current_max_per_row

    offsets_matrix_K = offsets_matrix_K + BLOCK_N*stride_kn
    offsets_matrix_V = offsets_matrix_V + BLOCK_N*stride_vn

  # Prevent division by zero and log of zero/negative values
  denominator_per_row_safe = tl.where(denominator_per_row > 0, denominator_per_row, 1e-6)
  accumulator = accumulator / denominator_per_row_safe[:, None]

  lse = prev_max_per_row + tl.log(denominator_per_row_safe)
  tl.store(LSE_ptr+offsets_m,lse,mask_q)

  offsets_matrix_O = offsets_m[:, None]* stride_om + offsets_d[None, :] * stride_od
  tl.store(O_ptr + offsets_matrix_O, accumulator.to(q_block.dtype), mask_q[: , None])


def flash_attention(
    Q : torch.Tensor,
    K : torch.Tensor,
    V : torch.Tensor,
    sm_scale : float = None
):
  assert Q.is_cuda and Q.is_contiguous()
  assert K.is_cuda and K.is_contiguous()
  assert V.is_cuda and V.is_contiguous()

  assert Q.shape[-1] == K.shape[-1]
  assert K.shape == V.shape

  B, H, T, C = Q.shape
  _, _, T2, _ = K.shape


  grid = lambda meta : (triton.cdiv(T, meta['BLOCK_M']),B*H)

  if sm_scale is None:
        sm_scale = 1.0 / (D ** 0.5)

  O = torch.empty_like(Q)
  LSE = torch.empty((B, H, T), dtype=torch.float32, device=Q.device)

  flash_attention_kernel[grid](
      Q,
      K,
      V,
      O,
      LSE,
      T, T2,
      Q.stride(2), Q.stride(3), Q.stride(0), Q.stride(1),
      K.stride(2), K.stride(3), K.stride(0), K.stride(1),
      V.stride(2), V.stride(3), V.stride(0), V.stride(1),
      O.stride(2), O.stride(3), O.stride(0), O.stride(1),
      LSE.stride(0), LSE.stride(1), LSE.stride(2),
      sm_scale,
      HEADS_PER_BATCH = H,
      BLOCK_M = 32,
      BLOCK_N = 32,
      WHOLE_D = C
    )

  return O, LSE

In [61]:
# The backward pass

@triton.jit
def flash_attention_backward_pre_kernel(
    O_ptr,
    dO_ptr,
    D_ptr,
    M,
    stride_ob, stride_oh, stride_om, stride_od,
    stride_dob, stride_doh, stride_dom, stride_dod,
    stride_db, stride_dh, stride_dm,
    HEAD_DATA: tl.constexpr,
    BLOCK_M: tl.constexpr,
    HEADS_PER_BATCH: tl.constexpr
):
  pid = tl.program_id(0)
  pid_bh = tl.program_id(1)

  current_batch = pid_bh // HEADS_PER_BATCH
  current_head = pid_bh % HEADS_PER_BATCH

  O_ptr = O_ptr + current_batch*stride_ob + current_head*stride_oh
  dO_ptr = dO_ptr + current_batch*stride_dob + current_head*stride_doh
  D_ptr = D_ptr + current_batch*stride_db + current_head*stride_dh

  offsets_m = pid*BLOCK_M + tl.arange(0,BLOCK_M)
  offsets_d = tl.arange(0,HEAD_DATA)

  offsets_matrix_o = offsets_m[:,None]*stride_om + offsets_d[None,:]*stride_od
  offsets_matrix_do = offsets_m[:,None]*stride_dom + offsets_d[None,:]*stride_dod

  mask_m = offsets_m < M

  O = tl.load(O_ptr+offsets_matrix_o, mask_m[:,None], other=0.0)
  dO = tl.load(dO_ptr+offsets_matrix_do, mask_m[:, None], other=0.0)

  D = tl.sum(O*dO, axis=1)

  tl.store(D_ptr + offsets_m*stride_dm, D, mask_m)


@triton.jit
def flash_attention_backward_kernel(
    Q_ptr, K_ptr, V_ptr, LSE_ptr, Delta_ptr, dO_ptr,
    dQ_ptr, dK_ptr, dV_ptr,
    M, N,
    sm_scale,
    stride_qb, stride_qh, stride_qm, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_dob, stride_doh, stride_dom, stride_dod,
    stride_dqb, stride_dqh, stride_dqm, stride_dqd,
    stride_dkb, stride_dkh, stride_dkn, stride_dkd,
    stride_dvb, stride_dvh, stride_dvn, stride_dvd,
    stride_lseb, stride_lseh, stride_lsem,
    stride_deltab, stride_deltah, stride_deltam,
    HEADS_PER_BATCH: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    HEAD_DIM: tl.constexpr,
):
  pid = tl.program_id(0)
  pid_bh = tl.program_id(1)

  current_batch = pid_bh // HEADS_PER_BATCH
  current_head = pid_bh % HEADS_PER_BATCH

  Q_ptr = Q_ptr + current_batch*stride_qb + current_head*stride_qh
  K_ptr = K_ptr + current_batch*stride_kb + current_head*stride_kh
  V_ptr = V_ptr + current_batch*stride_vb + current_head*stride_vh
  dO_ptr = dO_ptr + current_batch*stride_dob + current_head*stride_doh
  dQ_ptr = dQ_ptr + current_batch*stride_dqb + current_head*stride_dqh
  dK_ptr = dK_ptr + current_batch*stride_dkb + current_head*stride_dkh
  dV_ptr = dV_ptr + current_batch*stride_dvb + current_head*stride_dvh
  LSE_ptr = LSE_ptr + current_batch*stride_lseb + current_head*stride_lseh
  Delta_ptr = Delta_ptr + current_batch*stride_deltab + current_head*stride_deltah #causal

  offsets_m = tl.arange(0,BLOCK_M)
  offsets_n = pid*BLOCK_N + tl.arange(0,BLOCK_N)
  offsets_d = tl.arange(0, HEAD_DIM)

  offsets_matrix_q = offsets_m[:,None]*stride_qm + offsets_d[None,:]*stride_qd
  offsets_matrix_k = offsets_n[:, None]*stride_kn + offsets_d[None,:]*stride_kd
  offsets_matrix_v = offsets_n[:,None]*stride_vn + offsets_d[None,:]*stride_vd
  offsets_matrix_do = offsets_m[:,None]*stride_dom + offsets_d[None,:]*stride_dod
  offsets_matrix_dq = offsets_m[:,None]*stride_dqm + offsets_d[None,:]*stride_dqd
  offsets_matrix_dk = offsets_n[:,None]*stride_dkn + offsets_d[None,:]*stride_dkd
  offsets_matrix_dv = offsets_n[:,None]*stride_dvn + offsets_d[None,:]*stride_dvd
  offsets_matrix_lse = offsets_m*stride_lsem
  offsets_matrix_delta = offsets_m*stride_deltam

  mask_m = offsets_m < M
  mask_n = offsets_n < N

  k_block = tl.load(K_ptr + offsets_matrix_k, mask_n[:,None], other=0.0) # bN * D
  v_block = tl.load(V_ptr + offsets_matrix_v, mask_n[:,None], other=0.0) # bN * D
  dk_block = tl.zeros([BLOCK_N, HEAD_DIM], dtype=tl.float32)
  dv_block = tl.zeros([BLOCK_N, HEAD_DIM], dtype=tl.float32)

  for m in range(0, M, BLOCK_M):
    mask_m = offsets_m < (M-m)
    q_block = tl.load(Q_ptr + offsets_matrix_q, mask_m[:, None], other=0.0)
    do_block = tl.load(dO_ptr + offsets_matrix_do, mask_m[:, None], other=0.0)
    LSE_block = tl.load(LSE_ptr + offsets_matrix_lse, mask_m, other=-10000.0)
    Delta_block = tl.load(Delta_ptr + offsets_matrix_delta, mask_m, other=0.0)

    qk_block = tl.dot(q_block, tl.trans(k_block))*sm_scale

    # Prevent NaN values during exponentiation by clamping extremely low LSE values
    lse_clamped = tl.where(LSE_block > -1000.0, LSE_block, -1000.0)
    p_block = tl.exp(qk_block - lse_clamped[:, None])

    causal_mask = ((offsets_m + m)[:, None] >= (offsets_n)[None, :]) & mask_m[:, None] & mask_n[None, :]
    p_block = tl.where(causal_mask, p_block, 0.0)
    p_block = p_block.to(do_block.dtype)

    dv_block += tl.dot(tl.trans(p_block), do_block)

    dp_block = tl.dot(do_block, tl.trans(v_block))

    dqk_block = p_block * (dp_block - Delta_block[:, None]) * sm_scale
    dqk_block = tl.where(causal_mask, dqk_block, 0.0)
    dqk_block = dqk_block.to(k_block.dtype)

    dk_block += tl.dot(tl.trans(dqk_block) , q_block)

    sub_q_block = tl.dot(dqk_block , k_block)

    tl.atomic_add(dQ_ptr + offsets_matrix_dq, sub_q_block, mask_m[:, None])

    offsets_matrix_q += BLOCK_M*stride_qm
    offsets_matrix_dq += BLOCK_M*stride_dqm
    offsets_matrix_do += BLOCK_M*stride_dom
    offsets_matrix_lse += BLOCK_M*stride_lsem
    offsets_matrix_delta += BLOCK_M*stride_deltam

  tl.store(dK_ptr + offsets_matrix_dk, dk_block.to(k_block.dtype), mask=mask_n[:, None])
  tl.store(dV_ptr + offsets_matrix_dv, dv_block.to(v_block.dtype), mask=mask_n[:, None])


def flash_attention_backward(
    dO: torch.Tensor,
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    O: torch.Tensor,
    LSE: torch.Tensor,
    sm_scale: float = 0
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

    # Enforce contiguous layout
    dO = dO.contiguous()
    Q = Q.contiguous()
    K = K.contiguous()
    V = V.contiguous()
    O = O.contiguous()
    LSE = LSE.contiguous()

    B, H, M, D = Q.shape
    _, _, N, _ = K.shape

    if sm_scale is None:
        sm_scale = 1.0 / (D ** 0.5)

    Delta = torch.empty((B, H, M), device=Q.device, dtype=torch.float32)

    dQ = torch.zeros_like(Q, dtype=torch.float32)
    dK = torch.empty_like(K)
    dV = torch.empty_like(V)

    BLOCK_M = 32
    BLOCK_N = 16

    grid_pre = (triton.cdiv(M, BLOCK_M), B * H)

    flash_attention_backward_pre_kernel[grid_pre](
        O_ptr=O,
        dO_ptr=dO,
        D_ptr=Delta,
        M=M,
        stride_ob=O.stride(0), stride_oh=O.stride(1), stride_om=O.stride(2), stride_od=O.stride(3),
        stride_dob=dO.stride(0), stride_doh=dO.stride(1), stride_dom=dO.stride(2), stride_dod=dO.stride(3),
        stride_db=Delta.stride(0), stride_dh=Delta.stride(1), stride_dm=Delta.stride(2),
        HEAD_DATA=D,
        BLOCK_M=BLOCK_M,
        HEADS_PER_BATCH=H,
    )

    grid_bwd = (triton.cdiv(N, BLOCK_N), B * H)

    flash_attention_backward_kernel[grid_bwd](
        Q_ptr=Q, K_ptr=K, V_ptr=V,
        LSE_ptr=LSE, Delta_ptr=Delta, dO_ptr=dO,
        dQ_ptr=dQ, dK_ptr=dK, dV_ptr=dV,
        M=M, N=N,
        sm_scale=sm_scale,
        stride_qb=Q.stride(0), stride_qh=Q.stride(1), stride_qm=Q.stride(2), stride_qd=Q.stride(3),
        stride_kb=K.stride(0), stride_kh=K.stride(1), stride_kn=K.stride(2), stride_kd=K.stride(3),
        stride_vb=V.stride(0), stride_vh=V.stride(1), stride_vn=V.stride(2), stride_vd=V.stride(3),
        stride_dob=dO.stride(0), stride_doh=dO.stride(1), stride_dom=dO.stride(2), stride_dod=dO.stride(3),
        stride_dqb=dQ.stride(0), stride_dqh=dQ.stride(1), stride_dqm=dQ.stride(2), stride_dqd=dQ.stride(3),
        stride_dkb=dK.stride(0), stride_dkh=dK.stride(1), stride_dkn=dK.stride(2), stride_dkd=dK.stride(3),
        stride_dvb=dV.stride(0), stride_dvh=dV.stride(1), stride_dvn=dV.stride(2), stride_dvd=dV.stride(3),
        stride_lseb=LSE.stride(0), stride_lseh=LSE.stride(1), stride_lsem=LSE.stride(2),
        stride_deltab=Delta.stride(0), stride_deltah=Delta.stride(1), stride_deltam=Delta.stride(2),
        HEADS_PER_BATCH=H,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        HEAD_DIM=D,
    )

    return dQ, dK, dV

In [62]:
class custom_flash_attention(torch.autograd.Function):
  @staticmethod
  def forward(ctx, Q, K, V, sm_scale=None):
    #print('Forward_debug')
    if sm_scale is None:
      sm_scale = 1.0 / (Q.shape[-1] ** 0.5)
    O, LSE = flash_attention(Q, K, V, sm_scale)
    ctx.save_for_backward(Q, K, V, O, LSE)
    ctx.sm_scale = sm_scale
    return O

  @staticmethod
  def backward(ctx, dO):
    #print('backward_debug')
    Q, K, V, O, LSE = ctx.saved_tensors
    dQ, dK, dV = flash_attention_backward(
            dO=dO,
            Q=Q,
            K=K,
            V=V,
            O=O,
            LSE=LSE,
            sm_scale = ctx.sm_scale
        )
    return dQ, dK, dV, None

In [63]:

text = ''
vocab = []
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()
  vocab = sorted(list(set(text)))

vocab
vocab_length = len(vocab)

print(f'Voculabulary: {vocab}')
print(f'letter count: {vocab_length}')

Voculabulary: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
letter count: 65


In [64]:
char_to_token_dict = { c:t for t , c in enumerate(vocab)}
token_to_char_dict = { t:c for t, c in enumerate(vocab)}

print('Vocabs: \n')
print(char_to_token_dict)
print(token_to_char_dict)

encode = lambda t : [char_to_token_dict[c] for c in t]
print(f'\nEncoder test: {encode('hello guys')}')

decode = lambda s : ''.join([token_to_char_dict[t] for t in s])
print(f'\nDecoder test: {decode(encode('hello guys'))}')

Vocabs: 

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h'

In [65]:
data_tensor = torch.tensor(encode(text), dtype=torch.long)

data_len = len(data_tensor)

print(f'Data tensor of len: {data_len}')

split_count = int(0.8*data_len)
train_data = data_tensor[:split_count]
test_data = data_tensor[split_count:]

print(f'Train data size: {len(train_data)}')
print(f'Test data size: {len(test_data)}')

def get_batch(b_type):
  batch_pool = train_data if b_type == 'train' else test_data if b_type == 'test' else None
  if(batch_pool == None):
    print('INVALID BATCH TYPE, RETURNING -1')
    return -1, -1
  random_index = torch.randint(data_len - block_size, (batch_size,))
  x = torch.stack([data_tensor[i:i+block_size] for i in random_index]).to(device)
  y = torch.stack([data_tensor[i+1:i+block_size+1] for i in random_index]).to(device)
  return x,y

example_x, example_y  = get_batch('train')
print('\nTesting get batch')
print(f'Example x: {example_x}')
print(f'Example y: {example_y}')

Data tensor of len: 1115394
Train data size: 892315
Test data size: 223079

Testing get batch
Example x: tensor([[46,  1, 45,  ..., 46, 39, 50],
        [39, 58,  1,  ..., 59, 50, 42],
        [43, 59, 58,  ..., 53,  5, 57],
        ...,
        [47, 56,  1,  ...,  1, 40, 59],
        [58, 46,  1,  ..., 57, 12,  0],
        [45,  1, 39,  ...,  1, 63, 53]], device='cuda:0')
Example y: tensor([[ 1, 45, 59,  ..., 39, 50, 50],
        [58,  1, 49,  ..., 50, 42,  1],
        [59, 58, 43,  ...,  5, 57,  1],
        ...,
        [56,  1, 22,  ..., 40, 59, 58],
        [46,  1, 61,  ..., 12,  0, 13],
        [ 1, 39, 50,  ..., 63, 53, 59]], device='cuda:0')


In [66]:
class CausalAttentionLayer(nn.Module):
  def __init__(self, embed_count, head_count):
    super().__init__()
    assert embed_count % head_count == 0
    self.embed_count = embed_count
    self.head_count = head_count
    self.embed_per_head = embed_count // head_count # since this is a sinpler model, for each head of attention we will split the data

    self.qkv_op = nn.Linear(embed_count, 3*embed_count, bias= False) # instead of tediously mutliplying each embed with Q, K, V,we
    self.mix = nn.Linear(embed_count, embed_count, bias = False)
    self.dropout = nn.Dropout(dropout_percent)

  def forward(self, x):
    B, T, C = x.shape
    # B can be batch size or just 1 depending on use case
    # T is Token count, upto block size
    # C is always embed count

    QKV = self.qkv_op(x)
    Q, K, V = QKV.split(C, dim=2)
    # For each B, T, ill get a new set of vectors describing a question or key for every Token per batch. These can either be equal to embed count( like int his case)
    # WQ, WK, and WV have shape C, d_k. DOt product will be done per B
    # or it can be anything else u wish, provided they are all equal
    # All thre are shaped as B, T, d_k (d_k = C in this case)

    Q = Q.view(B, T, self.head_count, self.embed_per_head).transpose(1,2).contiguous()
    K = K.view(B, T, self.head_count, self.embed_per_head).transpose(1,2).contiguous()
    V = V.view(B, T, self.head_count, self.embed_per_head).transpose(1,2).contiguous()
    # Split for heads, then switch T nd H. Now shapes are B, H, T, d_k/H.

    attention_result = custom_flash_attention.apply(Q, K, V)
    #attention_result = F.scaled_dot_product_attention(Q, K, V, is_causal = True, dropout_p=dropout_percent if self.training else 0)

    # first we transpose K, so its shape becomes B, H, d_k/H, T
    # QK = A has shape B, H, T, T. This is our self attention matrix for each token with every other token
    # we will then mask it, applying -inf to upper triangle for causality
    # then we mutlipl by V who has shape B, H, T, d_k/H. Result will be B, H, T, d_k/h

    attention_result = attention_result.transpose(1, 2).contiguous().view(B, T, C)
    attention_result = self.mix(attention_result)
    attention_result = self.dropout(attention_result)
     # we reshape through view, merging all ehads togerht. B, T, d_k
    # We muliptly this with a a final matrix to make the conlsuons fo each head mix wiwth each other. Shape of the W is d_k * C
    # FInal result shape: B T C

    return attention_result

class FeedForward(nn.Module): # the intuition for the feed forward layers is "fact storage." Or mixing embed info with fact info
  def __init__(self, embed_count):
    super().__init__()
    self.embed_count = embed_count
    self.network = nn.Sequential(
        nn.Linear(embed_count, embed_count*4), # this should be projected over each embed vector
        # shape of input is B, T, C, Since Linear is amtric dot product + addition, It will be applied per batch.
        # weigth shape = C, 4C
        # result shape B, T, 4C
        nn.GELU(), # industry standard, B, T, 4c
        nn.Linear(embed_count*4, embed_count),
        # weight shape:  4C, C
        #result = B T C
        nn.Dropout(dropout_percent)
    )

  def forward(self, x):
    return self.network(x)

class Block(nn.Module):
  def __init__(self, embed_count, head_count):
    super().__init__()
    self.embed_count = embed_count
    self.head_count = head_count
    self.normalize_1 = nn.LayerNorm(embed_count)
    self.attention_layer = CausalAttentionLayer(embed_count, head_count)
    self.normalize_2 = nn.LayerNorm(embed_count)
    self.feed_forward_layer = FeedForward(embed_count)

  def forward(self, x):
    # no need for shape tracking, Eveything is B T C
    result = self.normalize_1(x)
    result = result + self.attention_layer(result)
    result = result + self.feed_forward_layer(self.normalize_2(result))
    return result

class MiniGPT(nn.Module):
  def __init__(self, vocab_count, embed_count, head_count, layer_count):
    super().__init__()
    self.vocab_count = vocab_count
    self.embed_count = embed_count
    self.head_count = head_count
    self.layer_count = layer_count

    self.embedder = nn.Embedding(vocab_count, embed_count)
    # generates indital embedded matrix
    self.pos_embedder = nn.Embedding(block_size, embed_count)
    self.block_sequence = nn.Sequential(*[Block(embed_count, head_count) for c in range(layer_count)])
    self.final_normalize = nn.LayerNorm(embed_count)
    self.unembed = nn.Linear(embed_count, vocab_count, bias = False)
    self.embedder.weight = self.unembed.weight

    self.apply(self.initialize_weights) # applies the inserted fucntion for every sub module

    for name, p in self.named_parameters():
          if name.endswith('mix.weight'):
              torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * layer_count))

  def initialize_weights(self, module): # to prevent exploding weights and gradients, start small
      if isinstance(module, nn.Linear):
          torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
          if module.bias is not None:
              torch.nn.init.zeros_(module.bias)
      elif isinstance(module, nn.Embedding):
          torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, x, target=None):
    B, T = x.shape # shape is B, T
    # B is batch count, T is token count/ block size
    x = self.embedder(x)
    # Now shape is B, T C, with C = Embed count
    position_embeds = self.pos_embedder(torch.arange(T, device=device))
    # positional embed is a matrix which embeds vector positon numbers to unique vectors
    # for each position it will return a unique fector of size C. For T tokens, Positon_embeds = T, C
    x = x + position_embeds # B T C matrix will be added with a T C matric, So the result is broadcasted over B Batches

    x = self.block_sequence(x) # Result is B T C
    x = self.final_normalize(x)
    logits = self.unembed(x)# # unembed is of shape C, Tt, giving a
    # Tehrfore, result shape is B T Tt. THis matrix gives the probability of every possibkle next token (Tt )for a given token(T) in the batch B

    if target is None:
      return logits, None #Final shape, B T Tt
    else:
      # Now hence forth, C = Tt here
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      # Cross entropy has very specific conditions for operations. IN general the shapes needed Ar M * N and M
      target = target.view(B*T)
      # Think of it as the last of correct token ids. Logits on the other hand has a list fo predicitons for eevry possible one per token
      loss = F.cross_entropy(logits, target)
      return logits, loss # final shape B*T, C

  def generate(self, full_context, new_tokens, top_k = 30, temperature = 0.8):
    for _ in range(new_tokens):
      #shape = B*T, B = 1 (Unsqueezed data)
      block_context = full_context[:, -block_size:]
      logits, _ = self.forward(block_context) # shape: B T C
      logits = logits[:, -1, :]/temperature # take the prediction for the last token for each batch (or in this case 1 batch), So the shape is B, C, with B = 1 usually (note:, we cans till have mutliple batches for generation too, no hard and fast rule, this is for my own visualization)

      # now we wanna take top k predictions, set the rest to - infitnity. They will then be made irrelevant when applying softmax

      if(top_k is not None):
        least_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)) ) # since logits is B C, least vals is B top_k
        logits[logits < least_vals[:, [-1]]] = float('-inf')# loast_vals[: [-1]] has shape B , 1

      preds = torch.softmax(logits, dim= -1) # Shape B, C, along C we now have probabilites. Vast majority are now 0
      next_token = torch.multinomial(preds, 1) # B, 1, for each batch it has the token id for the next best token. Note that since th last dimension is of form token id, the shape is now B, T, with T = 1
      full_context = torch.cat([full_context, next_token], dim=1) # B, T+1, Type B T
      generated_char = decode(next_token[0].tolist())
      print(generated_char, end='')
    return full_context


model = MiniGPT(vocab_length, 512, 4, 5).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 15,917,056


In [67]:


def cosine_lr_controller(lr, current_epoch, max_epochs):
  min_lr = lr*0.1
  warm_up_epochs = 50
  if(current_epoch< warm_up_epochs):
    return lr * (current_epoch+1)/warm_up_epochs # smooth rising of lr to max lr in a linear fashion. We add one so we never hve a 0 learning rate
  else:
    pi_coeff = (current_epoch-warm_up_epochs)/(max_epochs - warm_up_epochs) # coeff moves from 0 to 1 linearly as we move form wamr up epochs to max epochs
    # we will use this value to smoothly move down lr to min lr
    cos_coeff = 0.5 *(1+ math.cos(math.pi*pi_coeff)) # trasnsions from 1 to 0 around a smooth and graceful curve
    return (lr-min_lr)*cos_coeff + min_lr # smooth transion from lr to min_lr



In [68]:
@torch.no_grad()
def evaluate(model, iterations):
  outputs = []
  model.eval()
  for j, data_set in enumerate(['train', 'test']):
    losses = torch.zeros(iterations)
    for i in range(iterations):
      xb, yb = get_batch(data_set)
      logits, loss = model.forward(xb, yb)
      losses[i] = loss
    mean_loss = losses.mean().item()
    outputs.append(mean_loss)
  model.train()
  return outputs

learning_rate = 4e-4
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate, betas = (0.9, 0.95), weight_decay = 0.1)
epochs = 2500
model.train()

for epoch in range(epochs):
  effective_lr = cosine_lr_controller(learning_rate, epoch, epochs)
  for param_group in optimizer.param_groups:
    param_group['lr'] = effective_lr # research param groups later
  xb, yb = get_batch('train')
  logits, loss = model(xb, yb)
  print(f"Current epoch: {epoch}")
  if epoch % 25 == 0:
    losses = evaluate(model, 5)
    print('==============================================================================================================================')
    print(f'EVALUATION FOR EPOCH {epoch}, TRAIN LOSS: {losses[0]}, TEST LOSS: {losses[1]}')
    print('==============================================================================================================================')

  optimizer.zero_grad()
  loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), 1) # trunicates gradients to prevent massive gradient epxlosisosn
  optimizer.step()

checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item(),
}

torch.save(checkpoint, 'checkpoint.pt')

Current epoch: 0
EVALUATION FOR EPOCH 0, TRAIN LOSS: 6.198477745056152, TEST LOSS: 6.207158088684082
Current epoch: 1
Current epoch: 2
Current epoch: 3
Current epoch: 4
Current epoch: 5
Current epoch: 6
Current epoch: 7
Current epoch: 8
Current epoch: 9
Current epoch: 10
Current epoch: 11
Current epoch: 12
Current epoch: 13
Current epoch: 14
Current epoch: 15
Current epoch: 16
Current epoch: 17
Current epoch: 18
Current epoch: 19
Current epoch: 20
Current epoch: 21
Current epoch: 22
Current epoch: 23
Current epoch: 24
Current epoch: 25
EVALUATION FOR EPOCH 25, TRAIN LOSS: 3.1671485900878906, TEST LOSS: 3.181889533996582
Current epoch: 26
Current epoch: 27
Current epoch: 28
Current epoch: 29
Current epoch: 30
Current epoch: 31
Current epoch: 32
Current epoch: 33
Current epoch: 34
Current epoch: 35
Current epoch: 36
Current epoch: 37
Current epoch: 38
Current epoch: 39
Current epoch: 40
Current epoch: 41
Current epoch: 42
Current epoch: 43
Current epoch: 44
Current epoch: 45
Current epoc

KeyboardInterrupt: 

In [70]:
prompt = 'MARQUES: \nWhere thoust stay?'
model.eval()
context = torch.tensor(encode(prompt), dtype=torch.long, device=device) # shape is T

# notes to self
# Perofromance with built in self attenion at epoch 450 with chakespear data set 1.692
# Perofromance with built in my custom kernel at epoch 450 with chakespear data set 1.68
generated_chars = decode(model.generate(context.unsqueeze(0), new_tokens=10000, temperature=0.8, top_k=40)[0].tolist())


 for captains charms that offence
May be a braced by this virtue and hereafters,
Which it will encounter many one of sweet,
To wise suck and men to take of law,
And seeing the business of the duke mind.
I would not be; I'll be a mountain tears;
For pity on the tyrannous company;
Even do your tent lawful is that have to the
state is that appeal'd for a bark. This is a spirit
Me like it on my life, therefore comes to you;
And yet the city of my gracious lord.

PROSPERO:
Most word to my state, and do you unprivate him.

MIRANDA:
My son, sir; but so has you done to so death.

GREMIO:
He hath not to be your infit Laurence' coast.

BAPTISTA:
Say thou hadst not to his body, and chance to him,
Perditaby in padam the first of death.

PETRUCHIO:
Each of his better happy to say to the queen?

BAPTISTA:
How now! you have still heard for him.

BAPTISTA:
O marry, what 'shall be of your highness' access?

BAPTISTA:
If it be be contented.

BAPTISTA:
I knew the noble of your company.

PETRUCHIO:
A maid